# Lecture 03 — XGBoost: boosting as functional optimization

We will build an additive predictor one tree at a time, derive the regularized leaf values and split score, then evaluate the method on hourly bike-demand scenarios. The rows are independently generated scenarios, **not a chronological time series**. This lesson studies regression on those scenarios; it does not estimate performance on future dates.

After this lesson, you should be able to connect negative gradients to residual fitting, derive the second-order tree objective, explain what its regularizers do, and assess the result with a clean train/validation/test design.

![Teaching figure for xgboost](../../docs/figures/03_xgboost.svg)

*An actual shallow tree fitted to an illustrative one-dimensional sample. Notice constant predictions inside each leaf.*

## What is a regression tree?

A split asks a question such as `temperature_c <= 18`. It sends each row to one of two branches. Repeating splits partitions feature space into leaves, each of which returns a constant. Tree depth counts decisions along a path; a regression tree therefore produces a stepwise surface, not a smooth polynomial.

A random forest fits randomized trees and averages their predictions. Boosting adds trees sequentially to improve the current predictor. The forest below is a benchmark, not another derivation you need to master first.

**One leaf calculation:** with gradient sum $G=-6$, Hessian sum $H=3$, and $\lambda=1$, the unshrunk optimal leaf value is $-G/(H+\lambda)=1.5$. A learning rate of 0.1 adds 0.15 to predictions in that leaf. These numbers illustrate the formula; they are not diagnostics extracted from the trained model.

One-hot hour labels avoid imposing a linear order but do not encode that 23:00 and 00:00 are neighbors. Lecture 07 introduces sine/cosine calendar features for that different modeling choice.

## 1. From gradient descent to a sequence of trees

Suppose the current predictor is a function \(F_{m-1}(x)\). Functional gradient boosting adds a weak learner in a direction that reduces empirical risk:

\[
F_m(x)=F_{m-1}(x)+\eta f_m(x),\qquad
f_m \text{ chosen to align with }-\nabla_F L.
\]

For squared error, \(\ell(y,F)=\tfrac12(y-F)^2\), this convention gives Hessian \(h_i=1\); using \(\ell=(y-F)^2\) doubles both gradient and Hessian. The negative derivative with respect to the current prediction is \(y-F\). Thus a simple boosting algorithm fits each new tree to the current residuals. For a general differentiable loss, the analogous targets are pseudo-residuals \(-\partial \ell(y_i,F(x_i))/\partial F\).

Trees are not arbitrary vectors in a function space: they partition feature space into leaves and assign one value per leaf. XGBoost uses a second-order expansion of the objective to choose both the tree structure and those values. That curvature information makes the update more informative than plain residual fitting. The learning-rate \(\eta\) shrinks each stagewise update.

## 2. Deriving the regularized leaf value

At a boosting round, let \(g_i=\partial_{\hat y_i}\ell(y_i,\hat y_i)\) and \(h_i=\partial^2_{\hat y_i}\ell(y_i,\hat y_i)\), evaluated at the current predictions. For a proposed tree \(f\), a second-order Taylor approximation (dropping constants independent of \(f\)) is

\[
\widetilde{L}(f)=\sum_i\left(g_i f(x_i)+\tfrac12h_i f(x_i)^2\right)+\Omega(f),
\quad \Omega(f)=\gamma T+\tfrac12\lambda\sum_{j=1}^{T}w_j^2+\alpha\sum_{j=1}^{T}|w_j|.
\]

Here \(T\) is the number of leaves, \(w_j\) the leaf values, \(\gamma\) the per-leaf complexity charge, \(\lambda\ge0\) the L2 penalty, and \(\alpha\ge0\) the L1 penalty. A larger \(\gamma\) requires a larger improvement before a split is worthwhile.

For one fixed leaf \(j\), define \(G_j=\sum_{i\in I_j}g_i\) and \(H_j=\sum_{i\in I_j}h_i\). Without L1, the leaf-dependent term is \(G_jw_j+\tfrac12(H_j+\lambda)w_j^2\). Assume the aggregate curvature satisfies \(H_j+\lambda>0\), so this quadratic is strictly convex in \(w_j\). Differentiating gives

\[
w_j^*=-\frac{G_j}{H_j+\lambda}.
\]

With L1, the minimizer is the soft-thresholded value \(w_j^*=-\operatorname{sgn}(G_j)(|G_j|-\alpha)_+/(H_j+\lambda)\), where \((u)_+=\max(u,0)\). L1 can set a leaf update exactly to zero; L2 continuously shrinks it. These are local optima for the fixed tree partition under the Taylor approximation, not a global optimum of the full learning problem.

## 3. Split gain and its limits

For the common L2-only derivation, define the score of a leaf as \(S(G,H)=G^2/(H+\lambda)\). Splitting a parent into left and right children has approximate gain

\[
\operatorname{Gain}=\tfrac12\left[S(G_L,H_L)+S(G_R,H_R)-S(G_P,H_P)\right]-\gamma.
\]

With L1, replace each squared numerator by \((|G|-\alpha)_+^2\). XGBoost searches candidate feature thresholds and prefers splits with larger objective reduction, subject to constraints. `min_child_weight` requires a minimum sum of Hessians in each child; for squared error that sum is proportional to the number of rows. `max_depth`, `gamma`, `reg_lambda`, and `reg_alpha` restrict complexity in different ways.

This is an approximate training-objective reduction for a particular loss and current model. Positive gain alone does not promise better performance on new rows. Greedy split search, finite precision, constraints, and subsequent stages all matter. Tree regularization and validation answer different questions: regularization shapes fitted trees; validation checks the complete training procedure on held-out examples.

### Verify one regularized split by hand

These four illustrative gradients and unit Hessians are local Taylor quantities, not extracted diagnostics from the bike model. With L2 penalty 1, the parent gradient sums to zero. A proposed split separates negative and positive gradient sums. Check the gain after the split penalty and the optimal leaf values before applying a learning rate.

In [ ]:
import numpy as np

gradients = np.array([-2.0, -1.0, 1.0, 2.0])
hessians = np.ones(4)
l2, split_penalty = 1.0, 0.2
G_left, H_left = gradients[:2].sum(), hessians[:2].sum()
G_right, H_right = gradients[2:].sum(), hessians[2:].sum()
G_parent, H_parent = gradients.sum(), hessians.sum()
left_weight = -G_left / (H_left + l2)
right_weight = -G_right / (H_right + l2)
gain = 0.5 * (G_left**2 / (H_left + l2) + G_right**2 / (H_right + l2)
              - G_parent**2 / (H_parent + l2)) - split_penalty
print({"left_leaf": left_weight, "right_leaf": right_weight, "split_gain": gain})
assert np.isclose(gain, 2.8)


## 4. A small residual-tree boosting round by round

Before using XGBoost, we can see first-order boosting directly. For squared error, the negative gradient is the residual. We initialize with the training mean, fit a shallow regression tree to the current residuals, and add a shrunken fraction of its output. Repeating this produces a sequence of functions. The panels compare two learning rates at the same tree depth and number of rounds. This toy is a teaching aid: it uses residual fitting and does not implement XGBoost's second-order split search or regularization.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

TOY_SEED = 42
rng = np.random.default_rng(TOY_SEED)
toy_x_train = np.linspace(-3, 3, 100)
toy_y_train = (
    np.sin(1.4 * toy_x_train)
    + 0.25 * toy_x_train
    + rng.normal(0, 0.12, len(toy_x_train))
)
toy_x_grid = np.linspace(-3.2, 3.2, 400)
toy_truth_grid = np.sin(1.4 * toy_x_grid) + 0.25 * toy_x_grid
etas = (0.15, 0.8)
rounds = 8
toy_predictions = {}
toy_losses = {}
for eta in etas:
    initial = float(toy_y_train.mean())
    fitted_train = np.full_like(toy_y_train, initial)
    fitted_grid = np.full_like(toy_x_grid, initial)
    snapshots = {0: fitted_grid.copy()}
    losses = [np.mean((toy_y_train - fitted_train) ** 2)]
    for step in range(1, rounds + 1):
        residual = toy_y_train - fitted_train
        tree = DecisionTreeRegressor(
            max_depth=2, min_samples_leaf=6, random_state=TOY_SEED
        )
        tree.fit(toy_x_train.reshape(-1, 1), residual)
        fitted_train += eta * tree.predict(toy_x_train.reshape(-1, 1))
        fitted_grid += eta * tree.predict(toy_x_grid.reshape(-1, 1))
        losses.append(np.mean((toy_y_train - fitted_train) ** 2))
        if step in (1, 2, 4, 8):
            snapshots[step] = fitted_grid.copy()
    toy_predictions[eta] = snapshots
    toy_losses[eta] = losses

fig, axes = plt.subplots(
    1, 3, figsize=(16, 4.5)
)
for ax, eta in zip(axes[:2], etas):
    ax.scatter(
        toy_x_train, toy_y_train, s=13, alpha=0.4,
        label="noisy training data",
    )
    ax.plot(
        toy_x_grid, toy_truth_grid, color="black", linestyle="--",
        label="underlying curve",
    )
    stages = (0, 1, 2, 4, 8)
    colors = ("gray", "#4c78a8", "#f58518", "#54a24b", "#e45756")
    for step, color in zip(stages, colors):
        ax.plot(
            toy_x_grid, toy_predictions[eta][step],
            color=color, label=f"round {step}",
        )
    ax.set(
        title=f"Residual trees, eta={eta}",
        xlabel="x", ylabel="prediction",
    )
    ax.legend(fontsize=8)
for eta in etas:
    axes[2].plot(
        range(rounds + 1), toy_losses[eta], marker="o",
        label=f"eta={eta}",
    )
axes[2].set(
    title="Training squared error",
    xlabel="boosting round", ylabel="mean squared error",
)
axes[2].legend()
plt.tight_layout()


A larger learning rate takes bigger steps, so it may reduce training loss faster while also producing larger changes from one round to the next. With this fixed toy and only eight rounds, interpret the curves as an illustration of step size, not a universal ranking. The production experiment below uses XGBoost's second-order objective and stops using a separate validation set.


## 5. Load and inspect the scenarios

The data path helper searches the current directory and its parents for `data/bike_demand.csv`. The expected columns are the six predictors and `rentals` as the response.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

SEED = 42
FEATURES = ["temperature_c", "humidity", "wind_kmh", "hour", "weekday", "season"]
TARGET = "rentals"

def find_data_file(filename="bike_demand.csv"):
    # In Jupyter, cwd is commonly the repository root or this notebook's folder.
    for start in (Path.cwd(), *Path.cwd().parents):
        candidate = start / "data" / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find data/" + filename + " in the current directory or its parents. "
        "Open Jupyter from the repository or place the supplied CSV in data/."
    )

data_path = find_data_file()
df = pd.read_csv(data_path)
missing = sorted(set(FEATURES + [TARGET]) - set(df.columns))
if missing:
    raise ValueError(f"{data_path} is missing required columns: {missing}")
df = df[FEATURES + [TARGET]].dropna().copy()
print(f"Loaded {len(df):,} complete scenario rows from {data_path}")
df.head()


In [ ]:
df.describe(include="all").T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["rentals"].hist(bins=30, ax=axes[0], color="steelblue")
axes[0].set(title="Demand across generated scenarios", xlabel="rentals", ylabel="rows")
axes[1].scatter(df["temperature_c"], df["rentals"], alpha=0.25, s=12)
axes[1].set(title="Demand versus temperature", xlabel="temperature (°C)", ylabel="rentals")
plt.tight_layout()

Because the scenarios are independent samples, a random partition is suitable for this exercise. If these were a time series, this split would leak future regimes into training and would be the wrong evaluation design. We reserve 20% as a final test set, then reserve 18.75% of the remaining 80% (15% overall) as validation data. Early stopping will inspect validation RMSE; only the final comparison uses test rows.

In [ ]:
# Treat hour and weekday as labels, not measurements with meaningful linear distance.
# One-hot columns are shared across all models so the comparison uses the same inputs.
categorical = ["hour", "weekday", "season"]
numeric = ["temperature_c", "humidity", "wind_kmh"]
encoder = ColumnTransformer(
    [("numeric", "passthrough", numeric),
     ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)],
    verbose_feature_names_out=False,
)
X = df[FEATURES]
y = df[TARGET].astype(float)

# First hold out the final test set. Then split the remaining development rows
# into model-fit and early-stopping validation subsets.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.1875, random_state=SEED
)  # 0.1875 * 0.80 = 0.15 of all rows
print(f"train={len(X_train)}, validation={len(X_val)}, test={len(X_test)}")


## 6. Fit comparable models

Hour, weekday, and season are encoded as categories so the models do not have to infer that, for example, hour 23 is close to hour 0 from a linear numeric scale. We fit the encoder on training rows only. Every model uses the same encoded training and test rows. The validation rows are given to XGBoost only to decide when to stop adding trees.

In [ ]:
# Fit preprocessing only on training rows, then use that same representation everywhere.
X_train_enc = encoder.fit_transform(X_train)
X_val_enc = encoder.transform(X_val)
X_test_enc = encoder.transform(X_test)

dummy = DummyRegressor(strategy="mean").fit(X_train_enc, y_train)
forest = RandomForestRegressor(
    n_estimators=400, min_samples_leaf=2, max_features=0.9,
    random_state=SEED, n_jobs=-1,
).fit(X_train_enc, y_train)

# Validation data controls early stopping. The test data is untouched until scoring.
xgb = XGBRegressor(
    objective="reg:squarederror", eval_metric="rmse",
    n_estimators=2000, learning_rate=0.04, max_depth=4,
    min_child_weight=5, subsample=0.85, colsample_bytree=0.9,
    reg_lambda=3.0, reg_alpha=0.0, gamma=0.0,
    early_stopping_rounds=50,
    random_state=SEED, n_jobs=-1,
)
xgb.fit(X_train_enc, y_train, eval_set=[(X_val_enc, y_val)], verbose=False)
print("XGBoost best iteration:", xgb.best_iteration)


`n_estimators` is a generous upper limit. `early_stopping_rounds` is passed to the `XGBRegressor` constructor; the validation score must fail to improve for 50 rounds before fitting stops. `best_iteration` identifies the best validation round. This is model selection, so report validation as part of the process and keep the test score as a single final assessment.

In [ ]:
history = xgb.evals_result()["validation_0"]["rmse"]
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(history) + 1), history, label="validation RMSE")
plt.axvline(xgb.best_iteration + 1, color="black", linestyle="--", label="best iteration")
plt.xlabel("boosting round")
plt.ylabel("RMSE")
plt.title("Early stopping uses the validation partition")
plt.legend()
plt.tight_layout()

## 7. Evaluate once on the test partition

RMSE weights larger errors more strongly; MAE is easier to interpret as a typical absolute error; \(R^2\) measures improvement relative to a constant-mean predictor on this test sample. The baseline itself is trained on the same training rows.

In [ ]:
def metric_row(name, model):
    pred = model.predict(X_test_enc)
    return {"model": name,
            "RMSE": mean_squared_error(y_test, pred) ** 0.5,
            "MAE": mean_absolute_error(y_test, pred),
            "R²": r2_score(y_test, pred)}

results = pd.DataFrame([
    metric_row("Training-mean baseline", dummy),
    metric_row("Random forest", forest),
    metric_row("XGBoost", xgb),
]).reset_index(drop=True)
results


In [ ]:
# This diagnostic was chosen in advance; do not select a model using test scores.
pred = xgb.predict(X_test_enc)
plt.figure(figsize=(5, 5))
plt.scatter(y_test, pred, alpha=0.35, s=14)
limits = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(limits, limits, "k--", linewidth=1)
plt.xlabel("actual rentals")
plt.ylabel("XGBoost prediction")
plt.title("XGBoost test predictions")
plt.tight_layout()


## 8. Interpret, do not overclaim

Compare each score to the baseline and ask whether the reduction in error matters for the intended use. Inspect where residuals are largest and whether errors vary by season or time of day. This dataset supports comparison across generated scenarios. It cannot establish chronological forecasting skill, robustness to distribution shift, or causal effects. Trying many parameter settings against this same test partition would turn it into another validation set and make the reported score optimistic.

### Takeaways

1. Gradient boosting adds trees stage by stage to reduce a loss in function space.
2. XGBoost's second-order approximation yields regularized leaf weights and a split score.
3. `gamma`, L1/L2 penalties, depth, and child-weight constraints control different aspects of tree complexity.
4. Early stopping uses validation data; final test data remains separate until evaluation.
5. An independent scenario split does not measure future-time forecasting.